# LLM-as-a-Judge: Quick Lab

This short notebook teaches the core pattern behind judge-based evaluation: ask one model to evaluate model outputs against an explicit rubric, and require a structured result.

The travel-agent evaluation uses this same idea inside ADK-native metrics such as `final_response_match_v2`. Here we keep the example small so the main afternoon can focus on running and editing the native eval set.

By the end you should be able to explain:

- why exact text matching is brittle
- why judge prompts need clear criteria
- why structured judge outputs are easier to analyze
- why judge scores are useful signals, not absolute truth

## 1. Setup

In [ ]:
import json
from enum import Enum

import pandas as pd
from litellm import completion
from pydantic import BaseModel, Field

> Only if you are running outside GitHub Codespaces or Instruqt, uncomment the next cell so the notebook can read your local `.env` file.

In [ ]:
# from dotenv import load_dotenv
# load_dotenv()

## 2. Tiny Evaluation Set

We compare two candidate models on two prompts. That is intentionally small: the goal is to learn the pattern, not to draw a serious leaderboard conclusion.

In [ ]:
MODEL_CANDIDATE_A = "gemini/gemini-3.5-flash-lite"  
MODEL_CANDIDATE_B = "gemini/gemini-3.1-pro-preview"  
MODEL_JUDGE  = "gemini/gemini-3.5-flash"        

EVAL_DATASET = [
    {
        "prompt_id": "explain_photosynthesis",
        "prompt": "Explain photosynthesis in simple terms for a 5th grader.",
    },
    {
        "prompt_id": "coffee_slogans",
        "prompt": "Write three distinct marketing slogans for an eco-friendly coffee brand.",
    },
    {
        "prompt_id": "tricky_reasoning",
        "prompt": "A bat and a ball cost $1.10 together. The bat costs $1.00 more than the ball. How much does the ball cost? Show your reasoning in at most 2 sentences.",
    },
]


print(f"Candidate A: {MODEL_CANDIDATE_A}")
print(f"Candidate B: {MODEL_CANDIDATE_B}")
print(f"Judge      : {MODEL_JUDGE}")

pd.DataFrame(EVAL_DATASET)

## 3. Generate Candidate Responses

For each prompt, we ask both candidate models for an answer. In a real eval, you would usually cache these responses so the judge can be re-run without paying to regenerate everything.

In [ ]:
candidate_responses = []

for item in EVAL_DATASET:
    print(f"Generating responses for {item['prompt_id']}...")

    response_a = completion(
        model=MODEL_CANDIDATE_A,
        messages=[{"role": "user", "content": item["prompt"]}],
    )
    response_b = completion(
        model=MODEL_CANDIDATE_B,
        messages=[{"role": "user", "content": item["prompt"]}],
    )

    candidate_responses.append(
        {
            "prompt_id": item["prompt_id"],
            "prompt": item["prompt"],
            "response_a": response_a.choices[0].message.content,
            "response_b": response_b.choices[0].message.content,
        }
    )
pd.set_option("display.max_colwidth", None)
pd.DataFrame(candidate_responses)[["prompt_id", "prompt", "response_a", "response_b"]]

## 4. Define the Judge

A useful judge needs two things:

1. A clear rubric.
2. A structured output schema, so the result can be checked and summarized programmatically.

In [ ]:
class Winner(str, Enum):
    MODEL_A = "Model A"
    MODEL_B = "Model B"
    TIE = "Tie"
    BOTH_BAD = "Both Bad"


class Judgment(BaseModel):
    rationale: str = Field(description="Brief reason for the decision, grounded in the rubric.")
    score_a: int = Field(description="Score for Model A from 1 to 5.", ge=1, le=5)
    score_b: int = Field(description="Score for Model B from 1 to 5.", ge=1, le=5)
    winner: Winner = Field(description="Overall winner.")


JUDGE_SYSTEM_PROMPT = """\
You are an impartial evaluator comparing two model responses.

Use this rubric:
- Helpfulness: does the response answer the user's request?
- Relevance: does it stay on task?
- Clarity: is it easy to understand?
- Completeness: does it include the important details?

Prefer the better response, but choose Tie when both are similarly good.
Choose Both Bad when neither response is useful.
Return only valid JSON matching the requested schema.
"""

JUDGE_USER_TEMPLATE = """\
### User Prompt
{prompt}

### Model A Response
{response_a}

### Model B Response
{response_b}
"""

## 5. Run the Judge

The judge is itself an LLM, so results can vary. We use low temperature and structured output to reduce noise, but the score is still a judgment, not ground truth.

In [ ]:
pd.set_option("display.max_colwidth", None)

judgments = []

for row in candidate_responses:
    print(f"Judging {row['prompt_id']}...")
    judge_prompt = JUDGE_USER_TEMPLATE.format(
        prompt=row["prompt"],
        response_a=row["response_a"],
        response_b=row["response_b"],
    )

    judge_response = completion(
        model=MODEL_JUDGE,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": judge_prompt},
        ],
        response_format=Judgment,
        temperature=0.1,
    )

    judgment = Judgment(**json.loads(judge_response.choices[0].message.content))
    judgments.append({"prompt_id": row["prompt_id"], **judgment.model_dump()})

judgments_df = pd.DataFrame(judgments)
judgments_df

## 6. Summarize

With only two prompts, the summary is a classroom discussion starter, not a statistically meaningful benchmark.

In [ ]:
display(judgments_df)

print("Winner counts:")
print(judgments_df["winner"].value_counts())

print("\nAverage scores:")
print(f"Model A: {judgments_df['score_a'].mean():.2f}")
print(f"Model B: {judgments_df['score_b'].mean():.2f}")

## 7. Mini Exercises

> #### Exercise 2.1: Change the Rubric
>
> Add one criterion to `JUDGE_SYSTEM_PROMPT`, such as conciseness, creativity, or factual accuracy. Re-run sections 5 and 6. Did the winner change?

> #### Exercise 2.2: Add One Prompt
>
> Add one more prompt to `EVAL_DATASET`. Re-run sections 3, 5, and 6. Does the small sample still support the same conclusion?

> #### Exercise 2.3: Judge the Judge
>
> Read one rationale carefully. Do you agree with it? If not, what would you change in the rubric?

## 8. Bridge to Agent Evaluation

For normal LLM responses, the judge mostly reads text.

For agents, text is not enough. You also need to inspect the actions the agent took: which tools it called, with which arguments, in what order, and whether it did unnecessary work.

That is why the travel-agent evaluation uses ADK's native eval set:

```text
exercises/01_agents/02_travel_agent/travel_agent/travel_agent_eval.evalset.json
```

Each case stores both expectations:

- `intermediate_data.tool_uses` for the expected tool trajectory
- `final_response` for the expected user-facing answer

ADK's eval metrics include `tool_trajectory_avg_score` and `final_response_match_v2`, so you can evaluate actions and answers separately.